# 29 — Pair-rate split at low data levels (runs on the Mac)

**The question.** Real hospitals differ in how resistant a given bug is to a given drug (e.g. E. coli vs ciprofloxacin: 27% at one Gaza hospital, 57% at another). If our hospitals differ like that, does a *similarity fingerprint* help the server decide whom to average with? And does it help more when each hospital has very little data?

**What this notebook does, in plain steps**
1. **Builds the hospitals.** 10 equal hospitals in 3 regions. For every common bug–drug pair, each hospital's resistance rate is nudged up or down by a realistic amount (a 2–3.6× odds difference, the size seen in published hospital and regional comparisons). Drugs of one class move together; hospitals in the same region resemble each other. About 3.5% of the test results get their label changed to make this happen.
2. **Checks the hospitals are really different** (no training): prints the rates against the real-world numbers, and checks whether hospitals of the same region come out closer than hospitals of different regions.
3. **Gate at 3% data.** Trains only FedAvg and an *oracle* that averages exactly within the true region. If even the oracle can't beat FedAvg, nothing can, and we stop.
4. **Runs everything** at 10%, 3%, 1% and 0.3% of each hospital's data, 5 seeds: FedAvg, the topology fingerprint, the label fingerprint, each with a *shuffled* control (fingerprints handed to the wrong hospitals), and the oracle. Every arm gets the same small fine-tune at the end, and local-only training is the baseline.
5. **Reports** each comparison with 95% intervals, and plots the gain over local-only against the amount of data.

Rules for what counts as a win are fixed in `docs/2026-11-25-pair-rate-lowdata-preregistration.md`.

**Time and safety.** Training is on the Mac CPU and takes roughly an hour or two for everything. Every finished arm and seed is saved to `results/amr_pairrate_f*.json`, so you can stop and re-run any cell and it skips what is done. Only cells 5 and 6 train.

In [ ]:
# 1) Environment (Mac, CPU)
import os, sys, glob, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
if 'ARMD_DIR' not in os.environ:
    hits = glob.glob(os.path.expanduser('~/Library/CloudStorage/GoogleDrive-*/My Drive/ARMD'))
    if hits:
        os.environ['ARMD_DIR'] = hits[0]
import torch, flwr, ray
print('torch', torch.__version__, '| flwr', flwr.__version__, '| ray', ray.__version__, '| data:', os.environ.get('ARMD_DIR'))
import importlib.util
def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, ROOT / rel)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m
pr = load('pr', 'scripts/2026-11-24-pair-rate-split.py')
lp = load('lp', 'scripts/2026-11-25-pair-rate-lowdata.py')
print('levels:', lp.FRACS, '| seeds:', lp.SEEDS, '| arms:', list(lp.ARMS))

## 1. The hospitals
Every well-tested bug–drug pair gets a different resistance rate at each hospital, by a realistic amount.

In [ ]:
# 2) Build the split on the real cohort and compare with the real-world numbers
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
df2, pairs, flipped = pr.make_df(df)
pr.describe(df, df2, pairs, flipped)

## 2. Are some hospitals really alike? (no training)
Distance between hospitals of the same region vs different regions. A ratio below 1 means that measure can see the regions. A similarity method can only help if the regions are visible.

In [ ]:
# 3) Structure check
pr.structure_check(df2, pairs)

## 3. Gate: does knowing the true regions help? (trains FedAvg and the oracle at 3% data)
Pre-registered: the oracle must beat FedAvg + fine-tune with a 95% interval above zero. If not, stop here.

In [ ]:
# 4) Headroom gate (resume-safe)
passed = lp.headroom()

## 4. All levels and arms
Only run this if the gate passed. Skips whatever is already saved.

In [ ]:
# 5) Everything: 10%, 3%, 1%, 0.3%  x  FedAvg, topology, topology shuffled, label fingerprint, label shuffled, oracle
assert passed, 'gate failed: stop (see the pre-registration)'
lp.run()

## 5. Results
Unweighted mean test F1 over hospitals after fine-tuning, mean ± sd over 5 seeds, and paired differences with 95% intervals. A fingerprint only counts as working if it beats both FedAvg + fine-tune and its own shuffled control.

In [ ]:
# 6) Tables
lp.report()

In [ ]:
# 7) Figure: gain over local-only as each hospital keeps less data
import matplotlib.pyplot as plt
G = lp.gaps()
show = [('fedavg+ft', 'FedAvg + fine-tune', '#2a78d6'), ('v3_topo+ft', 'Topology fingerprint', '#eb6834'),
        ('v3_topo_shuffled+ft', 'Topology, shuffled', '#f0a98a'), ('v3_ab+ft', 'Label fingerprint', '#1baf7a'),
        ('v3_ab_shuffled+ft', 'Label, shuffled', '#9ed9c3'), ('oracle+ft', 'Oracle (true regions)', '#52514e')]
fig, ax = plt.subplots(figsize=(8, 4.6))
for k, lab, col in show:
    if k in G:
        xs = sorted(G[k]); ys = [G[k][x] for x in xs]
        ax.errorbar([x * 100 for x in xs], [y.mean() for y in ys], yerr=[y.std() for y in ys], fmt='-o', color=col, label=lab, capsize=3, lw=1.5, ms=5)
ax.axhline(0, color='#52514e', lw=1); ax.set_xscale('log')
ax.set_xticks([x * 100 for x in lp.FRACS]); ax.set_xticklabels([f'{x * 100:g}%' for x in lp.FRACS]); ax.minorticks_off()
ax.set_xlabel("Share of each hospital's data kept"); ax.set_ylabel('F1 gain over local-only (mean ± sd, 5 seeds)')
ax.set_title('Pair-rate split: gain over local-only vs amount of data', loc='left'); ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()